In [15]:
import pandas as pd
from collections import Counter

In [16]:
df = pd.read_csv("../data/paragraph_turns_full.csv")
df = df[['sentence', 'collectiveAction', 'racialJustice', 'sentence_length']]
df.head(1)

,sentence,collectiveAction,racialJustice,sentence_length
0,"[MUSIC] Welcome everyone to this Epic Life, th...",0,0,9


In [17]:
filenames = {
    "relief": "../data/annotation/processing_log_relief.txt",
    "pride": "../data/annotation/processing_log_pride.txt",
    "guilt": "../data/annotation/processing_log_guilt.txt",
    "excitement": "../data/annotation/processing_log_excitement.txt",
    "disappointment": "../data/annotation/processing_log_dissapointment.txt",  
}

label_counts = {}

for emotion, filepath in filenames.items():
    with open(filepath, "r") as f:
        labels = []
        for line in f:
            if "label=" in line:
                label = int(line.strip().split("label=")[-1])
                labels.append(label)
        label_counts[emotion] = Counter(labels)

for emotion, counts in label_counts.items():
    print(f"{emotion.capitalize()} label counts: {dict(counts)}")

Relief label counts: {0: 20170, 1: 3878}
Pride label counts: {1: 16689, 0: 120110, 75: 1}
Guilt label counts: {0: 32395, 1: 1013}
Excitement label counts: {0: 27389, 1: 4099}
Disappointment label counts: {0: 211656, 1: 22903, 75: 1}


In [18]:
def extract_labels(filepath):
    labels = {}
    with open(filepath, "r") as f:
        for line in f:
            if "Row" in line and "label=" in line:
                parts = line.strip().split("label=")
                row_num = int(parts[0].split()[1].replace(":", ""))
                label = int(parts[1])
                labels[row_num] = label
    return labels

In [19]:
emotion_labels = {emotion: extract_labels(path) for emotion, path in filenames.items()}

In [20]:
for emotion, labels_dict in emotion_labels.items():
    df[emotion] = df.index.map(lambda i: labels_dict.get(i, 0))  

In [21]:
df_filtered = df[df['sentence_length'] <= 10]
df_filtered = df_filtered[df_filtered['sentence_length'] > 2]
#df_collective_action = df_filtered[df_filtered['collectiveAction'] == 1]

In [ ]:
all_emotions = ['relief', 'pride', 'guilt', 'excitement', 'disappointment']
subset_df = df_filtered[df_filtered[all_emotions].any(axis=1)]
subset_df

,sentence,collectiveAction,racialJustice,sentence_length,relief,pride,guilt,excitement,disappointment
0,"[MUSIC] Welcome everyone to this Epic Life, th...",0,0,9,0,1,0,0,0
1,"Thank you so much for having me, I'm excited.",0,0,9,0,0,0,1,0
2,"Yes, we're excited too.",0,0,4,0,0,0,1,0
5,So I think that it was never a decision. I'm t...,0,0,9,0,1,0,0,0
18,"Like, well, I see it's different. You're not, ...",0,0,6,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...
234512,That is absolutely not going to happen now.,0,0,8,0,0,0,0,1
234519,That's a bummer.,0,0,3,0,0,0,0,1
234535,"Yeah, that's a bummer. And then the Dodgers ar...",0,0,4,0,0,0,0,1
234538,"Yeah, it's kind of sucks.",0,0,5,0,0,0,0,1


In [ ]:
# def sample_balanced_rows(df, emotion, n_per_type=5, seed=0, exclude_indices=set()):
#     pred_1 = df[(df[emotion] == 1) & (~df.index.isin(exclude_indices))]
#     pred_0 = df[(df[emotion] == 0) & (~df.index.isin(exclude_indices))]

#     # Sample from each label category (proxy for TP, FN, FP, TN)
#     tp_like = pred_1.sample(n=n_per_type, random_state=seed)
#     fn_like = pred_0.sample(n=n_per_type, random_state=seed + 1)
#     fp_like = pred_1.sample(n=n_per_type, random_state=seed + 2)
#     tn_like = pred_0.sample(n=n_per_type, random_state=seed + 3)

#     combined = pd.concat([tp_like, fn_like, fp_like, tn_like])
#     combined = combined.drop_duplicates()
#     combined["target_emotion"] = emotion
#     return combined

In [28]:
def sample_balanced_rows(df, emotion, seed=0, exclude_indices=set()):
    df = df[~df.index.isin(exclude_indices)]

    sampled_rows = []

    for ca_value in [0, 1]:
        subset = df[df['collectiveAction'] == ca_value]

        pred_1 = subset[subset[emotion] == 1]
        pred_0 = subset[subset[emotion] == 0]

        # For this CA value, take 5 from each predicted label
        pred_1_sample = pred_1.sample(n=5, random_state=seed + ca_value * 10 + 1)
        pred_0_sample = pred_0.sample(n=5, random_state=seed + ca_value * 10 + 2)

        sampled = pd.concat([pred_1_sample, pred_0_sample])
        sampled_rows.append(sampled)

    combined = pd.concat(sampled_rows).drop_duplicates()
    combined["target_emotion"] = emotion
    return combined

In [34]:
all_emotions = ['relief', 'pride', 'guilt', 'excitement', 'disappointment']
final_sample = pd.DataFrame()
used_indices = set()

for i, emotion in enumerate(all_emotions):
    sample = sample_balanced_rows(
        df_filtered,
        emotion,
        seed=200 + i,
        exclude_indices=used_indices
    )
    used_indices.update(sample.index)
    final_sample = pd.concat([final_sample, sample])

In [35]:
final_sample = final_sample.sample(frac=1, random_state=999).reset_index(drop=True)
final_sample[["sentence", "collectiveAction", "target_emotion"] + all_emotions].head(100)

,sentence,collectiveAction,target_emotion,relief,pride,guilt,excitement,disappointment
0,- Oh certainly.,0,relief,1,0,0,0,0
1,I think transcendental meditation was much big...,0,excitement,0,0,0,0,1
2,"Well said, my friend.",0,pride,0,0,0,0,0
3,But this is what's happening. Racial injustice...,1,pride,0,0,0,0,0
4,"Right now, people right around the world are p...",1,relief,0,0,0,0,0
...,...,...,...,...,...,...,...,...
95,This is a coordinated activity happening acros...,1,disappointment,0,1,0,0,1
96,- When I think about our power as melanated pe...,1,excitement,0,1,0,1,0
97,Black Lives Matter.,1,disappointment,0,1,0,0,0
98,And that's how children are seen. That's how c...,1,disappointment,0,0,0,0,1


In [36]:
final_sample[["sentence", "target_emotion"] + all_emotions].to_csv("../data/emotion_annotation_full.csv", index=False)